In [720]:
##Primero importamos pandas y numpy
import pandas as pd
import numpy as np
import re

In [721]:
#pip install xlrd

In [722]:
##Descargamos y miramos el archivo
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [723]:
##Miramos el tamaño de la tabla e info general 
df.shape

(7065, 23)

In [724]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7065 entries, 0 to 7064
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            7065 non-null   object 
 1   Year            7063 non-null   float64
 2   Type            7047 non-null   object 
 3   Country         7015 non-null   object 
 4   State           6578 non-null   object 
 5   Location        6498 non-null   object 
 6   Activity        6480 non-null   object 
 7   Name            6846 non-null   object 
 8   Sex             6486 non-null   object 
 9   Age             4070 non-null   object 
 10  Injury          7030 non-null   object 
 11  Fatal Y/N       6504 non-null   object 
 12  Time            3538 non-null   object 
 13  Species         3934 non-null   object 
 14  Source          7045 non-null   object 
 15  pdf             6799 non-null   object 
 16  href formula    6794 non-null   object 
 17  href            6796 non-null   o

### Cambiar los títulos a minúsculas y quitar espacios

In [725]:
df.columns = df.columns.str.lower().str.replace(" ","")
df.head()

,date,year,type,country,state,location,activity,name,sex,age,...,species,source,pdf,hrefformula,href,casenumber,casenumber.1,originalorder,unnamed:21,unnamed:22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Eliminamos las columnas sin datos

In [726]:
columns_to_drop = { 
    'pdf',
    'href',
    'hrefformula',
    'casenumber',
    'casenumber.1',
    'originalorder',
    'unnamed:21',
    'unnamed:22'
}
df = df.drop(columns=columns_to_drop)

### Eliminar duplicados

In [727]:
##Miramos los duplicados
df.duplicated().sum()

np.int64(1)

In [728]:
##Buscamos el único duplicado que existe, sobre todo porque es una muestra muy pequeña 
filas_duplicadas = df[df.duplicated(keep=False)]
filas_duplicadas

,date,year,type,country,state,location,activity,name,sex,age,injury,fataly/n,time,species,source
5436,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman
5437,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman


In [729]:
##Al verla, decidimoso eliminar la primera fila
df = df.drop(5436)

### Gestionamos los nulos

In [730]:
##Comprobamos cuántos nulos hay
df.isnull().sum()

date           0
year           2
type          18
country       50
state        487
location     567
activity     585
name         219
sex          579
age         2994
injury        35
fataly/n     561
time        3526
species     3131
source        20
dtype: int64

In [731]:
##Comprobamos el porcentaje de los datos para tomar las decisiones respecto a nuestras hipótesis
df.isnull().sum()/df.shape[0]

date        0.000000
year        0.000283
type        0.002548
country     0.007078
state       0.068941
location    0.080266
activity    0.082814
name        0.031002
sex         0.081965
age         0.423839
injury      0.004955
fataly/n    0.079417
time        0.499151
species     0.443233
source      0.002831
dtype: float64

In [732]:
## Decidimos en qué columnas vamos a trabajar basándonos en nuestras hipótesis: 
## 'Sex', 'Year', 'Type', 'Fatal', 'Date' y 'Country'
## Comenzamos a investigar qué tipos de datos nulos son y tomar decisiones: eliminar filas o imputar valores. 

### HIPÓTESIS 1 - 'SEX': ¿Mueren más hombres que mujeres?

In [733]:
##Creamos nuestro df específico para esta columna
df_sex = df[['sex']]

In [734]:
## Comprobamos que 'Sex' tiene valores nulos
df_sex.isnull().sum()/df.shape[0]

sex    0.081965
dtype: float64

In [735]:
##Eliminamos los valores nulos que tiene
df_sex = df_sex.dropna(subset=['sex'])

In [736]:
##Comprobamos cómo son los datos que tenemos, es decir, qué valores se han asociado a la serie. 
df_sex['sex'].unique()

array(['M', 'F', 'F ', 'M ', ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

In [737]:
## Nos damos cuenta de que hay valores que aunque no son nulos, no son válidos para hacer el análisis.
df_sex['sex'] = df_sex['sex'].str.strip().str.upper()

In [738]:
### Como solo hay cinco valores dierentes de 'M' y 'F' decidimos prescindir de ellos
df_sex['sex'].value_counts()

sex
M        5670
F         810
N           2
LLI         1
M X 2       1
.           1
Name: count, dtype: int64

In [739]:
## Vamos a filtrar todos los valores correctos
filtered_values = ['F', 'M']
df_sex = df_sex[df_sex['sex'].isin(filtered_values)]

In [755]:
df_sex['sex'].unique()

array(['M', 'F'], dtype=object)

### HIPÓTESIS 2 - 'YEAR': ¿En qué año hubo más muertes?

In [740]:
## Repetimos el mismo proceso que hicimos con 'sex'

In [741]:
df['year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [742]:
## Hacemos un análisis de los datos de year y decidimos quedarnos con los valores mayores de 1000. 
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df.loc[df['year'] <= 1000, 'year'] = np.nan

print("NaN:", df['year'].isna().sum())
print("Filas originales:", df.shape[0])

NaN: 134
Filas originales: 7064


In [743]:
#Eliminamos los nulos
df = df.dropna(subset=['year'])
print("Filas actuales", df.shape[0])

Filas actuales 6930


In [744]:
## Comprobamos si hay más nulos y si los datos están coherentes
df['year'].isnull().sum()

np.int64(0)

- Type

In [745]:
## Repetimos proceso
df['type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [746]:
#Decidimos que queremos que salgan en mayúsculas y sin espacios
df['type'] = df['type'].astype(str).str.strip().str.capitalize()

#Decidimos unificar el resto de valores que no son 'Provoked' o 'Unprovoked' a 'Unknown'
df['type'] = df['type'].replace({
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown'
})

#Decidimos que hay algunos que aunque no sean provocados, sí son causados por otras causas. Agrupamos esas causas al valor 'Other'
df['type'] = df['type'].replace({'Watercraft': 'Other', 'Sea disaster': 'Other', 'Boat': 'Other'})

In [747]:
#Comprobamos si los cambios se hicieron efectivos 
df['type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other'], dtype=object)

In [748]:
df['type'].isnull().sum()

np.int64(0)

- Fatal

In [749]:
## Repetimos proceso
df['fataly/n'].unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, 'Y x 2', ' N',
       'N ', 'y'], dtype=object)

In [750]:
#Quitamos espacios en blanco y ponemos todos en mayúscula, para unificar valores
df['fataly/n'] = df['fataly/n'].str.strip().str.upper()
df['fataly/n'].unique()

array(['N', 'Y', 'F', 'M', nan, 'NQ', 'UNKNOWN', 'Y X 2'], dtype=object)

In [751]:
df = df.dropna(subset=['fataly/n'])

In [752]:
filter_y = df['fataly/n'] == 'Y'
filter_n = df['fataly/n'] == 'N'
fatal_df = df[filter_y | filter_n]

In [753]:
fatal_df['fataly/n'].unique()

array(['N', 'Y'], dtype=object)